# CSMS AI Scoring Engine — VLLM Pipeline Evaluation

This notebook evaluates the VLLM Multi-Model AI Scoring Engine across all pipeline stages (Extraction, Vision LLM, Validation, AI Jurors, and Consensus Agent).

It supports:
1. **Single Category Testing (User Selectable)** on `test_submissions/marine` with all 8 agreed CSV reports.
2. **Full Batch Testing Across All Categories** on `test_submissions/marine` with all 8 agreed CSV reports.

## 1. Setup Environment & Force Reload Engine Modules

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
import importlib

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

# Force reload all core engine modules to guarantee latest updates are active
import engine.extractors.pdf_extractor
import engine.extractors.ocr_extractor
import engine.extractors.image_extractor
import engine.extractors.vision_extractor
import engine.extractors.router
import engine.providers.vllm_provider
import engine.jurors.compliance_juror
import engine.jurors.evidence_juror
import engine.jurors.hse_juror
import engine.consensus.consensus_agent
import engine.pipeline

for mod in [
    engine.extractors.pdf_extractor,
    engine.extractors.ocr_extractor,
    engine.extractors.image_extractor,
    engine.extractors.vision_extractor,
    engine.extractors.router,
    engine.providers.vllm_provider,
    engine.jurors.compliance_juror,
    engine.jurors.evidence_juror,
    engine.jurors.hse_juror,
    engine.consensus.consensus_agent,
    engine.pipeline,
]:
    importlib.reload(mod)

t0 = time.time()
from engine.providers.provider_factory import get_provider
print(f"[Step 1 Complete] All engine modules freshly reloaded in {time.time() - t0:.3f}s")

## 2. Load Model Roles & Verify REST Endpoints

In [ ]:
t1 = time.time()
roles_file = os.path.join(ROOT_DIR, "configs", "model_roles.json")
with open(roles_file, "r", encoding="utf-8") as f:
    model_roles = json.load(f)

print(f"Loaded Model Roles in {time.time() - t1:.3f}s:")
print(json.dumps(model_roles, indent=2))

t2 = time.time()
vision_prov = get_provider(role="vision_model")
reason_prov = get_provider(role="reasoning_model")
cons_prov = get_provider(role="consensus_model")

print(f"\n[Vision Provider]      Model: {vision_prov.model} | Endpoint: {vision_prov.chat_url}")
print(f"[Reasoning Provider]   Model: {reason_prov.model} | Endpoint: {reason_prov.chat_url}")
print(f"[Consensus Provider]   Model: {cons_prov.model} | Endpoint: {cons_prov.chat_url}")

## 3. Test One Selected Category on `test_submissions/marine` (User Selectable)

Change `SELECTED_CATEGORY` below to test any specific category folder or ID (e.g. `"8.1 Sertifikasi Sistem Pengelolaan K3LL"`, `"1 Kepemimpinan dan Komitmen"`, etc.).
This cell executes the full pipeline across all stages and prints the executive stage summary and discrete CSMS score (0, 3, 6, 10).

In [ ]:
t3 = time.time()
from engine.pipeline import score_category, discover_category_folders

# USER SELECTABLE CATEGORY (Default to 8.1 for fast pipeline evaluation):
SELECTED_CATEGORY = "7.1 Audit"

all_cats = discover_category_folders()
target_cat = None
for c in all_cats:
    if SELECTED_CATEGORY.lower() in c["folder_name"].lower() or SELECTED_CATEGORY.lower() == c["category_id"].lower():
        target_cat = c
        break

if not target_cat and all_cats:
    target_cat = all_cats[0]
    print(f"[INFO] Selected category '{SELECTED_CATEGORY}' not found. Defaulting to first discovered: {target_cat['folder_name']}")

if target_cat:
    print(f"=== EVALUATING SINGLE CATEGORY: {target_cat['category_id']} ({target_cat['folder_name']}) ===")
    print(f"Folder path: {target_cat['path']}\n")
    
    single_res = score_category(target_cat["category_id"], target_cat["path"])
    dur = time.time() - t3
    print(f"\n[Single Category Pipeline Execution Completed in {dur:.2f}s]")
    print(f"Category ID:      {single_res.get('category_id')}")
    print(f"Folder Name:      {single_res.get('folder_name')}")
    print(f"Predicted Score:  {single_res.get('predicted_score')} / 10 (Strict CSMS discrete scale 0, 3, 6, 10)")
    print(f"Expected Score:   {single_res.get('expected_score')} / 10")
    print(f"Confidence:       {single_res.get('confidence')}")
    print(f"Execution Status: {single_res.get('status').upper()}")
else:
    print("[WARN] No categories found in test_submissions/marine.")

## 4. Test All Categories on `test_submissions/marine` (Full Batch Evaluation)

This cell runs the entire AI scoring pipeline across every category folder in `test_submissions/marine` and generates all 8 comprehensive CSV reports combining results across all categories.

In [ ]:
t4 = time.time()
from engine.pipeline import evaluate_all_categories

print("=== STARTING FULL BATCH EVALUATION ON test_submissions/marine ===")
batch_report = evaluate_all_categories()
batch_dur = time.time() - t4
print(f"\n[Batch Evaluation Complete in {batch_dur:.2f}s]")
print(f"Total Categories Evaluated: {len(batch_report.get('categories', []))}")

## 5. Verify All 8 Agreed CSV Reports in `reports/`

In [ ]:
reports_dir = os.path.join(ROOT_DIR, "reports")
agreed_csvs = [
    "file_discovery.csv",
    "extraction_report.csv",
    "evidence_report.csv",
    "validator_report.csv",
    "reasoning_report.csv",
    "consensus_report.csv",
    "final_score_report.csv",
    "confusion_matrix.csv"
]

print("=== CHECKING 8 AGREED CSV REPORTS ===")
for r in agreed_csvs:
    p = os.path.join(reports_dir, r)
    exists = os.path.exists(p)
    size = os.path.getsize(p) if exists else 0
    print(f"- {r:<24} | {'EXISTS' if exists else 'MISSING':<8} | Size: {size} bytes")

## 6. Generate & Open Interactive HTML Dashboard (`viewer/index.html`)

In [ ]:
t6 = time.time()
viewer_dir = os.path.join(ROOT_DIR, "viewer")
from engine.viewer_generator import generate_html_viewer
generate_html_viewer(batch_report, viewer_dir)
index_path = os.path.join(viewer_dir, "index.html")
print(f"[Dashboard Generated] Saved to {index_path} in {time.time() - t6:.3f}s")

# Uncomment below to auto-launch browser:
# import webbrowser
# webbrowser.open('file://' + os.path.abspath(index_path))